**Lab Assignment - 5
Machine Learning with Scikit-learn and From Scratch**

**Part 1: Scikit-learn Preprocessing + Fixed Train-Test Split**

In [4]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Raw Data Loading
df = pd.read_csv("garments_worker_productivity.csv")

# Clean formatting issues and formulate targets
df["department"] = df["department"].astype(str).str.strip()
df["meets_target"] = (
    df["actual_productivity"] >= df["targeted_productivity"]
).astype(int)

# Separate features and targets
categorical_cols = ["quarter", "department", "day"]
numerical_cols = [
    "team",
    "targeted_productivity",
    "smv",
    "wip",
    "over_time",
    "incentive",
    "idle_time",
    "idle_men",
    "no_of_style_change",
    "no_of_workers",
]

X_raw = df[categorical_cols + numerical_cols].copy()
y_reg_raw = df["actual_productivity"].values
y_cls_raw = df["meets_target"].values

# 2. Preprocessing Pipeline Definition
num_pipe = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

cat_pipe = Pipeline(
    [
        (
            "onehot",
            OneHotEncoder(
                drop="first", sparse_output=False, handle_unknown="ignore"
            ),
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[("num", num_pipe, numerical_cols), ("cat", cat_pipe, categorical_cols)]
)

# Apply preprocessing to feature space
X_processed_sk = preprocessor.fit_transform(X_raw)

# 3. Train-Test Split (Single fixed split for exact reproducibility across parts)
(
    X_train_sk,
    X_test_sk,
    y_reg_train,
    y_reg_test,
    y_cls_train,
    y_cls_test,
    train_idx,
    test_idx,
) = train_test_split(
    X_processed_sk,
    y_reg_raw,
    y_cls_raw,
    np.arange(len(df)),
    test_size=0.2,
    random_state=42,
)

print(
    f"Preprocessed Feature Matrix Shape: {X_processed_sk.shape}"
)  # (1197, 21)
print(f"Training Samples: {X_train_sk.shape[0]}, Test Samples: {X_test_sk.shape[0]}")

Preprocessed Feature Matrix Shape: (1197, 20)
Training Samples: 957, Test Samples: 240


**Part 2: Train LinearRegression & LogisticRegression (Scikit-learn)**

In [5]:
import time
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)

# =====================================================================
# 1. Scikit-learn Linear Regression (actual_productivity)
# =====================================================================
t0 = time.perf_counter()
sk_reg = LinearRegression()
sk_reg.fit(X_train_sk, y_reg_train)
sk_reg_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_reg_sk = sk_reg.predict(X_test_sk)
sk_reg_pred_time = time.perf_counter() - t0

sk_mae = mean_absolute_error(y_reg_test, y_pred_reg_sk)
sk_rmse = np.sqrt(mean_squared_error(y_reg_test, y_pred_reg_sk))
sk_r2 = r2_score(y_reg_test, y_pred_reg_sk)

# =====================================================================
# 2. Scikit-learn Logistic Regression (meets_target)
# =====================================================================
t0 = time.perf_counter()
sk_cls = LogisticRegression(max_iter=1000, random_state=42)
sk_cls.fit(X_train_sk, y_cls_train)
sk_cls_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_cls_sk = sk_cls.predict(X_test_sk)
sk_cls_pred_time = time.perf_counter() - t0

sk_acc = accuracy_score(y_cls_test, y_pred_cls_sk)
sk_prec = precision_score(y_cls_test, y_pred_cls_sk)
sk_rec = recall_score(y_cls_test, y_pred_cls_sk)
sk_f1 = f1_score(y_cls_test, y_pred_cls_sk)

print("--- Scikit-Learn Regression Metrics ---")
print(f"MAE: {sk_mae:.4f} | RMSE: {sk_rmse:.4f} | R2: {sk_r2:.4f}")
print(
    f"Train Time: {sk_reg_train_time * 1000:.3f} ms | Prediction Time: {sk_reg_pred_time * 1000:.3f} ms"
)

print("\n--- Scikit-Learn Classification Metrics ---")
print(
    f"Accuracy: {sk_acc * 100:.2f}% | Precision: {sk_prec:.4f} | Recall: {sk_rec:.4f} | F1: {sk_f1:.4f}"
)
print(
    f"Train Time: {sk_cls_train_time * 1000:.3f} ms | Prediction Time: {sk_cls_pred_time * 1000:.3f} ms"
)

--- Scikit-Learn Regression Metrics ---
MAE: 0.1088 | RMSE: 0.1481 | R2: 0.1736
Train Time: 3.817 ms | Prediction Time: 0.313 ms

--- Scikit-Learn Classification Metrics ---
Accuracy: 73.75% | Precision: 0.7591 | Recall: 0.9435 | F1: 0.8413
Train Time: 11.921 ms | Prediction Time: 0.327 ms


**Part 3: Manual Implementation (NumPy + Pandas ONLY — no Scikit-learn)**

In [6]:
import time
import numpy as np
import pandas as pd

# -------------------------------------------------------------
# 1. Raw Data
# -------------------------------------------------------------
raw_df = pd.read_csv("garments_worker_productivity.csv")
raw_df["department"] = raw_df["department"].astype(str).str.strip()
raw_df["meets_target"] = (
    raw_df["actual_productivity"] >= raw_df["targeted_productivity"]
).astype(int)

# -------------------------------------------------------------
# 2. Manual Preprocessing Class
# -------------------------------------------------------------
class ManualPreprocessor:

    def __init__(self, num_cols, cat_cols):
        self.num_cols = num_cols
        self.cat_cols = cat_cols
        self.medians = {}
        self.means = {}
        self.stds = {}
        self.ohe_columns = []

    def fit_transform(self, df):
        df_work = df.copy()

        # Handle missing values (median) & compute normalization statistics
        for col in self.num_cols:
            self.medians[col] = df_work[col].median()
            df_work[col] = df_work[col].fillna(self.medians[col])
            self.means[col] = df_work[col].mean()
            std = df_work[col].std(ddof=0)  # matching population std of scaler
            self.stds[col] = 1.0 if std == 0 else std
            df_work[col] = (df_work[col] - self.means[col]) / self.stds[col]

        # Categorical One-Hot Encoding (drop first column for linear independence)
        ohe_df = pd.get_dummies(
            df_work[self.cat_cols], drop_first=True, dtype=float
        )
        self.ohe_columns = ohe_df.columns.tolist()

        return np.hstack([df_work[self.num_cols].values, ohe_df.values])

    def transform(self, df):
        df_work = df.copy()

        for col in self.num_cols:
            df_work[col] = df_work[col].fillna(self.medians[col])
            df_work[col] = (df_work[col] - self.means[col]) / self.stds[col]

        ohe_df = pd.get_dummies(
            df_work[self.cat_cols], drop_first=True, dtype=float
        )
        ohe_df = ohe_df.reindex(columns=self.ohe_columns, fill_value=0.0)

        return np.hstack([df_work[self.num_cols].values, ohe_df.values])


# Fit & transform features
num_cols = [
    "team",
    "targeted_productivity",
    "smv",
    "wip",
    "over_time",
    "incentive",
    "idle_time",
    "idle_men",
    "no_of_style_change",
    "no_of_workers",
]
cat_cols = ["quarter", "department", "day"]

manual_prep = ManualPreprocessor(num_cols, cat_cols)
X_scratch_all = manual_prep.fit_transform(raw_df[cat_cols + num_cols])
y_reg_all = raw_df["actual_productivity"].values
y_cls_all = raw_df["meets_target"].values

# -------------------------------------------------------------
# 3. Train-Test Split (reusing identical train/test index partition)
# -------------------------------------------------------------
X_train_scratch = X_scratch_all[train_idx]
X_test_scratch = X_scratch_all[test_idx]
y_reg_tr = y_reg_all[train_idx]
y_reg_te = y_reg_all[test_idx]
y_cls_tr = y_cls_all[train_idx]
y_cls_te = y_cls_all[test_idx]

# Add bias intercept (column of 1s)
X_tr_bias = np.hstack([np.ones((X_train_scratch.shape[0], 1)), X_train_scratch])
X_te_bias = np.hstack([np.ones((X_test_scratch.shape[0], 1)), X_test_scratch])

# -------------------------------------------------------------
# 4. Model Training From Scratch
# -------------------------------------------------------------
# A. Linear Regression (Closed-Form Solution: w = (X^T X)^-1 X^T y)
class ScratchLinearRegression:

    def __init__(self):
        self.w = None

    def fit(self, X, y):
        self.w = np.linalg.pinv(X.T @ X) @ X.T @ y

    def predict(self, X):
        return X @ self.w


# B. Logistic Regression (Vectorized Gradient Descent with Sigmoid)
class ScratchLogisticRegression:

    def __init__(self, lr=0.1, epochs=2500):
        self.lr = lr
        self.epochs = epochs
        self.w = None

    @staticmethod
    def _sigmoid(z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -250, 250)))

    def fit(self, X, y):
        m, n = X.shape
        self.w = np.zeros(n)
        for _ in range(self.epochs):
            preds = self._sigmoid(X @ self.w)
            grad = (X.T @ (preds - y)) / m
            self.w -= self.lr * grad

    def predict_proba(self, X):
        return self._sigmoid(X @ self.w)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


# Execute Linear Regression Training & Prediction
t0 = time.perf_counter()
manual_reg = ScratchLinearRegression()
manual_reg.fit(X_tr_bias, y_reg_tr)
sc_reg_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_reg_scratch = manual_reg.predict(X_te_bias)
sc_reg_pred_time = time.perf_counter() - t0

# Execute Logistic Regression Training & Prediction
t0 = time.perf_counter()
manual_cls = ScratchLogisticRegression(lr=0.1, epochs=2500)
manual_cls.fit(X_tr_bias, y_cls_tr)
sc_cls_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_cls_scratch = manual_cls.predict(X_te_bias)
sc_cls_pred_time = time.perf_counter() - t0

# -------------------------------------------------------------
# 5. Pure NumPy Evaluation Metrics
# -------------------------------------------------------------
def manual_regression_metrics(y_true, y_pred):
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    ss_res = np.sum((y_true - y_pred) ** 2)
    r2 = 1.0 - (ss_res / ss_tot)
    return mae, rmse, r2


def manual_classification_metrics(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tn = np.sum((y_true == 0) & (y_pred == 0))

    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )
    return accuracy, precision, recall, f1


sc_mae, sc_rmse, sc_r2 = manual_regression_metrics(y_reg_te, y_pred_reg_scratch)
sc_acc, sc_prec, sc_rec, sc_f1 = manual_classification_metrics(
    y_cls_te, y_pred_cls_scratch
)

print("--- Manual Scratch Regression Metrics ---")
print(f"MAE: {sc_mae:.4f} | RMSE: {sc_rmse:.4f} | R2: {sc_r2:.4f}")
print("\n--- Manual Scratch Classification Metrics ---")
print(
    f"Accuracy: {sc_acc * 100:.2f}% | Precision: {sc_prec:.4f} | Recall: {sc_rec:.4f} | F1: {sc_f1:.4f}"
)

--- Manual Scratch Regression Metrics ---
MAE: 0.1088 | RMSE: 0.1481 | R2: 0.1736

--- Manual Scratch Classification Metrics ---
Accuracy: 75.00% | Precision: 0.7647 | Recall: 0.9548 | F1: 0.8492


**Part 4: Comparison Tables + Optimization**

In [7]:
# =====================================================================
# PART C: ADVANCED MANUAL OPTIMIZATION (NUMPY & PANDAS ONLY)
# =====================================================================


def engineer_advanced_features(df):
    f_df = df.copy()

    # 1. Missingness indicator for WIP (retains domain signal of unmonitored items)
    f_df["wip_is_missing"] = f_df["wip"].isna().astype(float)
    f_df["wip_imputed"] = f_df["wip"].fillna(f_df["wip"].median())

    # 2. Logarithmic stabilization of heavily skewed variables
    f_df["log_incentive"] = np.log1p(np.maximum(0, f_df["incentive"]))
    f_df["log_wip"] = np.log1p(np.maximum(0, f_df["wip_imputed"]))
    f_df["log_overtime"] = np.log1p(np.maximum(0, f_df["over_time"]))

    # 3. Domain interactions & non-linear features
    f_df["ot_per_worker"] = f_df["over_time"] / (f_df["no_of_workers"] + 1e-5)
    f_df["target_incentive_inter"] = (
        f_df["targeted_productivity"] * f_df["log_incentive"]
    )
    f_df["target_sq"] = f_df["targeted_productivity"] ** 2

    return f_df


# Feature engineering
opt_raw = engineer_advanced_features(raw_df)
opt_num_cols = [
    "team",
    "targeted_productivity",
    "target_sq",
    "smv",
    "wip_imputed",
    "wip_is_missing",
    "log_wip",
    "log_incentive",
    "log_overtime",
    "ot_per_worker",
    "target_incentive_inter",
    "idle_time",
    "idle_men",
    "no_of_style_change",
    "no_of_workers",
]

# Manual Standard Normalization on continuous features
opt_means = opt_raw[opt_num_cols].iloc[train_idx].mean()
opt_stds = (
    opt_raw[opt_num_cols]
    .iloc[train_idx]
    .std(ddof=0)
    .replace(0, 1.0)
)

opt_num_scaled = (opt_raw[opt_num_cols] - opt_means) / opt_stds
opt_cat_encoded = pd.get_dummies(opt_raw[cat_cols], drop_first=True, dtype=float)

X_opt_matrix = np.hstack(
    [
        np.ones((len(opt_raw), 1)),  # bias intercept
        opt_num_scaled.values,
        opt_cat_encoded.values,
    ]
)

# Extract identical train and test partitions
X_tr_opt = X_opt_matrix[train_idx]
X_te_opt = X_opt_matrix[test_idx]

# -------------------------------------------------------------
# Optimized Regression: Closed-form L2 Ridge Regularization
# -------------------------------------------------------------
t0 = time.perf_counter()
lambda_reg = 5.0
I_reg = np.eye(X_tr_opt.shape[1])
I_reg[0, 0] = 0.0  # Do not penalize bias
w_ridge = np.linalg.solve(
    X_tr_opt.T @ X_tr_opt + lambda_reg * I_reg, X_tr_opt.T @ y_reg_tr
)
opt_reg_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_reg_opt = X_te_opt @ w_ridge
opt_reg_pred_time = time.perf_counter() - t0

opt_mae, opt_rmse, opt_r2 = manual_regression_metrics(
    y_reg_te, y_pred_reg_opt
)


# -------------------------------------------------------------
# Optimized Classification: Vectorized L2 Logistic Regression + Calibrated Cutoff
# -------------------------------------------------------------
class OptimizedLogisticRegression:

    def __init__(self, lr=0.12, reg=0.8, epochs=3500):
        self.lr = lr
        self.reg = reg
        self.epochs = epochs
        self.w = None

    def fit(self, X, y):
        m, n = X.shape
        self.w = np.zeros(n)
        for _ in range(self.epochs):
            p = 1.0 / (1.0 + np.exp(-np.clip(X @ self.w, -250, 250)))
            grad = (X.T @ (p - y)) / m
            grad[1:] += (self.reg / m) * self.w[1:]  # L2 weight decay on weights
            self.w -= self.lr * grad

    def predict_proba(self, X):
        return 1.0 / (1.0 + np.exp(-np.clip(X @ self.w, -250, 250)))


t0 = time.perf_counter()
opt_cls = OptimizedLogisticRegression(lr=0.12, reg=0.8, epochs=3500)
opt_cls.fit(X_tr_opt, y_cls_tr)
opt_cls_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
probs_opt = opt_cls.predict_proba(X_te_opt)
y_pred_cls_opt = (probs_opt >= 0.55).astype(
    int
)  # Calibrated probability decision threshold
opt_cls_pred_time = time.perf_counter() - t0

opt_acc, opt_prec, opt_rec, opt_f1 = manual_classification_metrics(
    y_cls_te, y_pred_cls_opt
)

# =====================================================================
# COMPACT COMPARISON TABLES (SCIKIT-LEARN VS MANUAL SCRATCH VS OPTIMIZED)
# =====================================================================
df_reg_comp = pd.DataFrame(
    {
        "Implementation": [
            "Scikit-learn (LinearRegression)",
            "Manual Scratch (OLS Closed-Form)",
            "Optimized Manual (Ridge + Feature Eng)",
        ],
        "MAE": [f"{sk_mae:.4f}", f"{sc_mae:.4f}", f"{opt_mae:.4f}"],
        "RMSE": [f"{sk_rmse:.4f}", f"{sc_rmse:.4f}", f"{opt_rmse:.4f}"],
        "R² Score": [f"{sk_r2:.4f}", f"{sc_r2:.4f}", f"{opt_r2:.4f}"],
        "Train Time (s)": [
            f"{sk_reg_train_time:.5f}",
            f"{sc_reg_train_time:.5f}",
            f"{opt_reg_train_time:.5f}",
        ],
        "Pred Time (s)": [
            f"{sk_reg_pred_time:.5f}",
            f"{sc_reg_pred_time:.5f}",
            f"{opt_reg_pred_time:.5f}",
        ],
    }
)

df_cls_comp = pd.DataFrame(
    {
        "Implementation": [
            "Scikit-learn (LogisticRegression)",
            "Manual Scratch (Vanilla GD)",
            "Optimized Manual (L2 + Calibrated Threshold)",
        ],
        "Accuracy": [
            f"{sk_acc*100:.2f}%",
            f"{sc_acc*100:.2f}%",
            f"{opt_acc*100:.2f}%",
        ],
        "Precision": [
            f"{sk_prec:.4f}",
            f"{sc_prec:.4f}",
            f"{opt_prec:.4f}",
        ],
        "Recall": [f"{sk_rec:.4f}", f"{sc_rec:.4f}", f"{opt_rec:.4f}"],
        "F1-Score": [f"{sk_f1:.4f}", f"{sc_f1:.4f}", f"{opt_f1:.4f}"],
        "Train Time (s)": [
            f"{sk_cls_train_time:.5f}",
            f"{sc_cls_train_time:.5f}",
            f"{opt_cls_train_time:.5f}",
        ],
        "Pred Time (s)": [
            f"{sk_cls_pred_time:.5f}",
            f"{sc_cls_pred_time:.5f}",
            f"{opt_cls_pred_time:.5f}",
        ],
    }
)

print("\n" + "=" * 80)
print("TABLE 1: REGRESSION BENCHMARK COMPARISON (Target: actual_productivity)")
print("=" * 80)
print(df_reg_comp.to_string(index=False))

print("\n" + "=" * 80)
print("TABLE 2: CLASSIFICATION BENCHMARK COMPARISON (Target: Meets Target)")
print("=" * 80)
print(df_cls_comp.to_string(index=False))


TABLE 1: REGRESSION BENCHMARK COMPARISON (Target: actual_productivity)
                        Implementation    MAE   RMSE R² Score Train Time (s) Pred Time (s)
       Scikit-learn (LinearRegression) 0.1088 0.1481   0.1736        0.00382       0.00031
      Manual Scratch (OLS Closed-Form) 0.1088 0.1481   0.1736        0.00662       0.00005
Optimized Manual (Ridge + Feature Eng) 0.0984 0.1349   0.3148        0.00158       0.00008

TABLE 2: CLASSIFICATION BENCHMARK COMPARISON (Target: Meets Target)
                              Implementation Accuracy Precision Recall F1-Score Train Time (s) Pred Time (s)
           Scikit-learn (LogisticRegression)   73.75%    0.7591 0.9435   0.8413        0.01192       0.00033
                 Manual Scratch (Vanilla GD)   75.00%    0.7647 0.9548   0.8492        0.07749       0.00008
Optimized Manual (L2 + Calibrated Threshold)   78.33%    0.8342 0.8814   0.8571        0.11947       0.00011
